In [1]:
from pyspark.sql import SparkSession
from pathlib import Path

In [2]:
jar_filename = "sqlite-jdbc-3.53.0.0.jar"
jar_path = Path(jar_filename)
jar_path.exists()

True

In [3]:
str(jar_path.resolve())

'/home/jovyan/work/sqlite-jdbc-3.53.0.0.jar'

In [ ]:
spark = (
    SparkSession.builder
    .appName("Northwind-OLAP-Spark")
    .config("spark.jars", str(jar_path.resolve()))
    # Desativa a Web UI para economizar memória e evitar loops de porta
    # .config("spark.ui.enabled", "false")
    # Força o modo local com apenas 1 thread para teste inicial
    # .master("local[1]")
    # Resolve o problema de Hostname do Docker
    # .config("spark.driver.host", "127.0.0.1")
    # .config("spark.driver.bindAddress", "127.0.0.1")
    # Garante que a JVM não tente alocar mais do que o container permite
    # .config("spark.driver.memory", "512m")
    .getOrCreate()
)

print("Spark Session iniciada com sucesso!")

In [ ]:
df_fact = (
    spark.read.format("jdbc")
    .option("url", "jdbc:sqlite:/home/jovyan/work/northwind-dw.db")
    .option("dbtable", "fact_sales")
    .load()
)

In [ ]:
df_fact.show(5)

In [ ]:
import os
import sys
from pyspark.sql import SparkSession

# 1. Path and existence check
jar_path = "/home/jovyan/work/sqlite-jdbc-3.53.0.0.jar" # Rename your file to this
if not os.path.exists(jar_path):
    print(f"CRITICAL: JAR not found at {jar_path}")

# 2. Force Python to use the internal Docker IP
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

spark = (
    SparkSession.builder
    .appName("Northwind-OLAP-Spark")
    .master("local[1]") # Use only 1 core to stabilize
    .config("spark.jars", jar_path)
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    # Low memory allocation to avoid JVM overhead crashes
    .config("spark.driver.memory", "1g")
    .config("spark.sql.shuffle.partitions", "1")
    # Set log level to see what is happening
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print("SUCCESS: Spark is alive.")